# 02 — Feature Engineering
## MIMIC-IV ICU Length-of-Stay Project

Input: `data/processed/feature_table.csv` (frozen output of the SQL pipeline,
see `docs/sql_pipeline_complete.md`). This notebook does **not** touch the
SQL pipeline and does **not** regenerate any feature — it only encodes,
organizes, and reviews the 17 features already extracted, then saves
`data/processed/model_ready.csv` for the modeling notebook
(`03_modeling.ipynb`).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "feature_table.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "model_ready.csv"

df = pd.read_csv(DATA_PATH)
print(f"Loaded {DATA_PATH.name} -> shape {df.shape}")

Loaded feature_table.csv -> shape (117, 19)


## Part 1.1 — Separate Variables by Type

In [2]:
NUMERICAL_COLS = [
    "age_at_admission", "first_creatinine", "first_wbc",
    "first_hemoglobin", "medication_count", "comorbidity_count",
]
CATEGORICAL_COLS = ["gender", "admission_type", "insurance"]
BINARY_COLS = [
    "weekend_admission", "night_admission",
    "diabetes_flag", "hypertension_flag", "ckd_flag", "heart_failure_flag",
    "antibiotic_flag", "insulin_flag",
]
TARGET_COL = "prolonged_stay_label"
KEY_COL = "stay_id"

print(f"Numerical ({len(NUMERICAL_COLS)}):   {NUMERICAL_COLS}")
print(f"Categorical ({len(CATEGORICAL_COLS)}): {CATEGORICAL_COLS}")
print(f"Binary ({len(BINARY_COLS)}):      {BINARY_COLS}")
print(f"Target:            {TARGET_COL}")
print(f"Key:               {KEY_COL}")

assert set([KEY_COL, TARGET_COL] + NUMERICAL_COLS + CATEGORICAL_COLS + BINARY_COLS) == set(df.columns)
print("\nOK - classification covers every column exactly once.")

Numerical (6):   ['age_at_admission', 'first_creatinine', 'first_wbc', 'first_hemoglobin', 'medication_count', 'comorbidity_count']
Categorical (3): ['gender', 'admission_type', 'insurance']
Binary (8):      ['weekend_admission', 'night_admission', 'diabetes_flag', 'hypertension_flag', 'ckd_flag', 'heart_failure_flag', 'antibiotic_flag', 'insulin_flag']
Target:            prolonged_stay_label
Key:               stay_id

OK - classification covers every column exactly once.


## Part 1.2 — One-Hot Encode Categorical Variables

In [3]:
# drop_first=True: avoids the dummy-variable trap (perfect collinearity with
# the model intercept) for Logistic Regression; harmless for tree models,
# which do not use an intercept and are invariant to this choice.
cat_dummies = pd.get_dummies(df[CATEGORICAL_COLS], drop_first=True).astype(int)
print(f"{len(CATEGORICAL_COLS)} categorical columns -> {cat_dummies.shape[1]} dummy columns:")
for c in cat_dummies.columns:
    print(" -", c)
cat_dummies.head()

3 categorical columns -> 8 dummy columns:
 - gender_M
 - admission_type_ELECTIVE
 - admission_type_EW EMER.
 - admission_type_OBSERVATION ADMIT
 - admission_type_SURGICAL SAME DAY ADMISSION
 - admission_type_URGENT
 - insurance_Medicare
 - insurance_Other


,gender_M,admission_type_ELECTIVE,admission_type_EW EMER.,admission_type_OBSERVATION ADMIT,admission_type_SURGICAL SAME DAY ADMISSION,admission_type_URGENT,insurance_Medicare,insurance_Other
0,1,0,0,0,0,1,0,0
1,1,0,0,0,0,1,0,0
2,1,0,0,0,0,1,0,0
3,0,0,0,0,0,1,0,1
4,1,0,0,0,0,1,1,0


## Part 1.3 — Binary Variables (Kept Unchanged)

In [4]:
for c in BINARY_COLS:
    bad = set(df[c].unique()) - {0, 1}
    assert not bad, f"{c} has non-binary values: {bad}"
print(f"All {len(BINARY_COLS)} binary flags confirmed in {{0,1}} — passed through unchanged, no encoding needed.")
df[BINARY_COLS].describe().T[["mean", "min", "max"]]

All 8 binary flags confirmed in {0,1} — passed through unchanged, no encoding needed.


,mean,min,max
weekend_admission,0.239316,0.0,1.0
night_admission,0.358974,0.0,1.0
diabetes_flag,0.367521,0.0,1.0
hypertension_flag,0.641026,0.0,1.0
ckd_flag,0.239316,0.0,1.0
heart_failure_flag,0.264957,0.0,1.0
antibiotic_flag,0.658120,0.0,1.0
insulin_flag,0.512821,0.0,1.0


## Part 1.4 — Numerical Variables: Scaling Strategy

**Design decision.** `model_ready.csv` stores the numerical features
**unscaled**. Standardization for Logistic Regression is applied later, in
`03_modeling.ipynb`, *inside* a `scikit-learn` `Pipeline`/`ColumnTransformer`
that is fit only on the training fold of each split.

This is a deliberate choice, not an omission: fitting a `StandardScaler` on
the full 117-row table (train + test combined) before the split — which is
what saving a pre-scaled column here would require — leaks the test set's
mean and standard deviation into training-time preprocessing. That is
exactly the class of subtle leakage this project has guarded against since
the SQL design stage (`docs/data_dictionary.md`, `docs/validation_strategy.md`
principles). Keeping `model_ready.csv` unscaled satisfies both halves of the
requirement at once: it *is* the "unscaled version for tree-based models,"
and it is also the correct starting point for a properly-fit scaler for
Logistic Regression — the two are the same artifact, not two separate files.

## Part 1.5 — Multicollinearity Review: Comorbidity Representation

In [5]:
comorbid_flags = ["diabetes_flag", "hypertension_flag", "ckd_flag", "heart_failure_flag"]
print("Correlation of comorbidity_count with its 4 component flags:")
print(df[comorbid_flags + ["comorbidity_count"]].corr()["comorbidity_count"].drop("comorbidity_count").round(3))

Correlation of comorbidity_count with its 4 component flags:
diabetes_flag         0.519
hypertension_flag     0.669
ckd_flag              0.685
heart_failure_flag    0.670
Name: comorbidity_count, dtype: float64


In [6]:
# Variance Inflation Factor (VIF) for each representation, in the context of
# the other numeric + binary features (excludes the 3 categorical dummies to
# keep this check focused on the comorbidity-representation question).
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

other_numeric = ["age_at_admission", "first_creatinine", "first_wbc", "first_hemoglobin", "medication_count"]
other_binary = ["weekend_admission", "night_admission", "antibiotic_flag", "insulin_flag"]

def vif_table(cols):
    X = sm.add_constant(df[cols])
    return pd.Series(
        [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
        index=X.columns,
    ).drop("const").round(2)

vif_a = vif_table(other_numeric + comorbid_flags + other_binary)
vif_b = vif_table(other_numeric + ["comorbidity_count"] + other_binary)

print("VIF — Representation A (individual comorbidity flags):")
print(vif_a.to_string())
print(f"  max VIF = {vif_a.max():.2f}")
print("\nVIF — Representation B (comorbidity_count):")
print(vif_b.to_string())
print(f"  max VIF = {vif_b.max():.2f}")

VIF — Representation A (individual comorbidity flags):
age_at_admission      1.52
first_creatinine      1.50
first_wbc             1.20
first_hemoglobin      1.09
medication_count      2.02
diabetes_flag         1.29
hypertension_flag     1.58
ckd_flag              1.98
heart_failure_flag    1.27
weekend_admission     1.11
night_admission       1.18
antibiotic_flag       1.35
insulin_flag          1.89
  max VIF = 2.02

VIF — Representation B (comorbidity_count):
age_at_admission     1.35
first_creatinine     1.26
first_wbc            1.07
first_hemoglobin     1.08
medication_count     1.86
comorbidity_count    1.48
weekend_admission    1.04
night_admission      1.14
antibiotic_flag      1.34
insulin_flag         1.57
  max VIF = 1.86


In [7]:
# Empirical check: quick 5-fold CV ROC-AUC, representation A vs B, using a
# regularized Logistic Regression (the model family most sensitive to
# collinearity). Not a tuning exercise -- a sanity check on the choice.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y = df["prolonged_stay_label"]

pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))])

Xa = df[other_numeric + comorbid_flags + other_binary]
scores_a = cross_val_score(pipe, Xa, y, cv=skf, scoring="roc_auc")

Xb = df[other_numeric + ["comorbidity_count"] + other_binary]
scores_b = cross_val_score(pipe, Xb, y, cv=skf, scoring="roc_auc")

print(f"Representation A (flags)             CV ROC-AUC: mean={scores_a.mean():.3f} std={scores_a.std():.3f}  folds={scores_a.round(3)}")
print(f"Representation B (comorbidity_count) CV ROC-AUC: mean={scores_b.mean():.3f} std={scores_b.std():.3f}  folds={scores_b.round(3)}")

Representation A (flags)             CV ROC-AUC: mean=0.528 std=0.150  folds=[0.741 0.42  0.676 0.431 0.373]
Representation B (comorbidity_count) CV ROC-AUC: mean=0.453 std=0.080  folds=[0.565 0.496 0.48  0.343 0.382]


**Decision: keep the individual comorbidity flags (Representation A), drop
`comorbidity_count`.**

- **Multicollinearity is not actually severe in either representation** —
  every VIF stays under 2.0, far below the conventional 5-10 concern
  threshold. The high pairwise correlation seen in the EDA (r≈0.52-0.69,
  `reports/EDA_summary.md` §2) reflects that `comorbidity_count` is
  *mechanically* the sum of these four flags, not that the flags are
  collinear with each other — so this is not a "must drop one or the model
  breaks" situation.
- **Information content favors the flags.** `comorbidity_count` is a lossy,
  deterministic summary of the four flags — it can always be recomputed from
  them, but not the reverse (a count of 2 doesn't say *which* two
  conditions). The flags are a strict superset of the information in the
  count.
- **Interpretability for the next project phase.** The proposal's
  explainability goal (SHAP, `proposal.docx` §2) is best served by
  per-condition attribution — "CKD contributed X to this prediction" is more
  actionable than "comorbidity burden contributed X."
- **Empirical check agrees, with the sample-size caveat stated plainly.** The
  quick CV comparison favors Representation A (mean ROC-AUC above) over B,
  though at n=117 with 5 folds this gap is within noise and is a
  confirmatory data point, not the primary justification.

`comorbidity_count` is therefore excluded from `model_ready.csv`; the four
individual flags (already part of `BINARY_COLS`) are retained unchanged.

## Assemble and Save `model_ready.csv`

In [8]:
FINAL_NUMERIC_COLS = [c for c in NUMERICAL_COLS if c != "comorbidity_count"]

model_df = pd.concat(
    [
        df[[KEY_COL, TARGET_COL]],
        df[FINAL_NUMERIC_COLS],
        cat_dummies,
        df[BINARY_COLS],
    ],
    axis=1,
)

print("model_ready shape:", model_df.shape)
print("\nColumns:")
for c in model_df.columns:
    print(" -", c, f"({model_df[c].dtype})")

model_ready shape: (117, 23)

Columns:
 - stay_id (int64)
 - prolonged_stay_label (int64)
 - age_at_admission (int64)
 - first_creatinine (float64)
 - first_wbc (float64)
 - first_hemoglobin (float64)
 - medication_count (int64)
 - gender_M (int32)
 - admission_type_ELECTIVE (int32)
 - admission_type_EW EMER. (int32)
 - admission_type_OBSERVATION ADMIT (int32)
 - admission_type_SURGICAL SAME DAY ADMISSION (int32)
 - admission_type_URGENT (int32)
 - insurance_Medicare (int32)
 - insurance_Other (int32)
 - weekend_admission (int64)
 - night_admission (int64)
 - diabetes_flag (int64)
 - hypertension_flag (int64)
 - ckd_flag (int64)
 - heart_failure_flag (int64)
 - antibiotic_flag (int64)
 - insulin_flag (int64)


In [9]:
# Validation before saving
assert model_df.isnull().sum().sum() == 0, "Unexpected missing values after encoding."
assert model_df[KEY_COL].is_unique, "stay_id is not unique after encoding."
assert set(model_df[TARGET_COL].unique()).issubset({0, 1}), "Target contains unexpected values."
assert model_df.shape[0] == df.shape[0], "Row count changed during feature engineering — it must not."
print(f"OK - {model_df.shape[0]} rows (unchanged from feature_table.csv), "
      f"{model_df.shape[1]} columns, 0 missing values, stay_id unique, target valid.")

model_df.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved: {OUTPUT_PATH.relative_to(PROJECT_ROOT)}")

OK - 117 rows (unchanged from feature_table.csv), 23 columns, 0 missing values, stay_id unique, target valid.

Saved: data\processed\model_ready.csv


In [10]:
print("Feature engineering completed successfully.")

Feature engineering completed successfully.
